Import Libraries and Configure Logging

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser
from sklearn.feature_extraction.text import TfidfVectorizer
from rake_nltk import Rake, Metric
from collections import Counter
from tqdm import tqdm
import logging
import random
import os
from scipy.stats import entropy

# Setup logging
logging.basicConfig(
    filename='lda_output.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.getLogger('gensim').setLevel(logging.WARNING)

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

Initialize NLP Tools and Stopwords

In [2]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Domain-specific stopwords
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma'
    'player', 'meeting', 'lab', 'staff', 'member', 'nursing', 'clinical', 'unix', 'palm', 'either'
}
stop_words.update(custom_stopwords)

# Review-specific stopwords
review_stopwords = {
    'review', 'star', 'rating', 'good', 'great', 'course', 'learn', 'learning',
    'would', 'like', 'could', 'one', 'bit', 'week', 'think', 'much', 'really',
    'lot', 'new', 'thank', 'thanks', 'many', 'well', 'also', 'get', 'time',
    'truly', 'even', 'make', 'see', 'content', 'material', 'class', 'work',
    'way', 'understand', 'information', 'helpful', 'useful', 'knowledge',
    'day', 'help', 'easy'
}
stop_words.update(review_stopwords)

Load PROMISE Data

In [3]:
def load_promise_data(file_path):
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Successfully loaded PROMISE data from {file_path}")
        return df
    except FileNotFoundError:
        logging.error(f"PROMISE data file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading PROMISE data: {e}")
        raise

promise_data_path = '../../datasets/PROMISE_exp_cleaned.csv'
promise_df = load_promise_data(promise_data_path)

Aggregate Data by Category

In [4]:
logging.info("Started aggregating PROMISE data by category")
category_texts = promise_df.groupby('_class_')['cleaned_text'].apply(lambda x: ' '.join(x)).to_dict()
for category, text in category_texts.items():
    word_count = len(text.split())
    logging.info(f"Category {category}: {word_count} words")
    print(f"Category {category}: {word_count} words")
    if word_count < 50:
        logging.warning(f"Category {category} has short text")
        print(f"Warning: Category {category} has short text ({word_count} words). Consider reviewing data.")
    if word_count == 0:
        logging.error(f"Category {category} has empty text")
        raise ValueError(f"Category {category} has empty text. Check data filtering.")

Category F: 3041 words
Category FT: 129 words
Category PE: 559 words
Category PO: 58 words
Category SC: 144 words
Category SE: 951 words
Category US: 653 words


Extract Seed Words

In [5]:
logging.info("Started extracting seed words")
seed_words = {}
target_word_count = 20
category_word_scores = {}

Extract RAKE phrases per document

In [6]:
logging.info("Started extracting RAKE phrases for individual documents")
document_phrases = []
for idx, row in tqdm(promise_df.iterrows(), total=len(promise_df), desc="Processing documents"):
    text = row['cleaned_text']
    category = row['_class_']
    
    # Skip empty or invalid texts
    if not isinstance(text, str) or len(text.strip()) == 0:
        logging.warning(f"Empty or invalid text at index {idx} for category {category}")
        continue
    
    # Initialize RAKE
    rake = Rake(
        stopwords=stop_words,
        min_length=2,
        max_length=2,
        ranking_metric=Metric.DEGREE_TO_FREQUENCY_RATIO,
        include_repeated_phrases=False
    )
    
    # Extract phrases
    rake.extract_keywords_from_text(text)
    phrases = rake.get_ranked_phrases_with_scores()
    
    # Store phrases with document index and category
    for score, phrase in phrases:
        document_phrases.append({
            'doc_id': idx,
            'category': category,
            'phrase': phrase,
            'score': score
        })
    
    # Log progress
    logging.info(f"Processed document {idx} in category {category}: {len(phrases)} phrases extracted")

phrases_df = pd.DataFrame(document_phrases)
logging.info(f"Total phrases extracted across all documents: {len(phrases_df)}")

Processing documents: 100%|██████████| 773/773 [00:00<00:00, 6311.14it/s]


Combine TF-IDF and RAKE for seed words

In [7]:
for category, text in category_texts.items():
    # TF-IDF Extraction (focus on single terms)
    vectorizer = TfidfVectorizer(
        max_features=200,
        ngram_range=(1, 1),  # Only unigrams
        stop_words=list(stop_words),
        min_df=1,
        sublinear_tf=True
    )
    tfidf_matrix = vectorizer.fit_transform([text])
    tfidf_terms = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.toarray()[0]
    tfidf_term_scores = {term: score for term, score in zip(tfidf_terms, tfidf_scores) if score > 0.03}
    logging.info(f"Category {category}: {len(tfidf_term_scores)} TF-IDF single terms extracted")

    # RAKE Phrase Extraction for Category (focus on phrases)
    category_phrases = phrases_df[phrases_df['category'] == category][['phrase', 'score']]
    aggregated_phrases = category_phrases.groupby('phrase').agg({'score': 'sum'}).reset_index()
    aggregated_phrases = aggregated_phrases.sort_values(by='score', ascending=False)
    logging.info(f"Category {category}: {len(aggregated_phrases)} RAKE phrases aggregated")

    # Convert RAKE phrases to terms (only multi-word phrases)
    rake_terms = []
    rake_term_scores = {}
    for _, row in aggregated_phrases.iterrows():
        phrase = row['phrase']
        score = row['score']
        words = [lemmatizer.lemmatize(word) for word in phrase.split()]
        if len(words) > 1:  # Only include multi-word phrases
            phrase_term = '_'.join(words)  # Join phrases with underscores
            rake_terms.append(phrase_term)
            rake_term_scores[phrase_term] = score
    logging.info(f"Category {category}: {len(rake_terms)} RAKE phrase terms extracted")

    # Combine TF-IDF (single terms) and RAKE (phrases)
    combined_terms = list(set(tfidf_terms) | set(rake_terms))
    logging.info(f"Category {category}: {len(combined_terms)} combined terms before filtering")

    # Filter terms to ensure single words from TF-IDF and phrases from RAKE
    filtered_terms = [
        term for term in combined_terms
        if term not in stop_words
        and len(term.replace('_', '')) > 2  # Allow short phrases
        and (
            ('_' not in term and tfidf_term_scores.get(term, 0) > 0.03) or  # Single words from TF-IDF
            ('_' in term and term in rake_term_scores)  # Phrases from RAKE
        )
    ]
    logging.info(f"Category {category}: {len(filtered_terms)} terms after filtering")

    # Score terms
    term_scores = {}
    word_freq = Counter(text.split())
    total_freq = sum(word_freq.values())
    max_rake_score = aggregated_phrases['score'].max() if not aggregated_phrases.empty else 1.0
    for term in filtered_terms:
        tfidf_score = tfidf_term_scores.get(term, 0)
        rake_score = rake_term_scores.get(term, 0)
        combined_score = 0.7 * tfidf_score + 0.3 * (rake_score / max_rake_score)
        term_scores[term] = combined_score
        logging.info(f"Category {category}: Term '{term}' - TF-IDF: {tfidf_score:.4f}, RAKE: {rake_score:.4f}, Combined: {combined_score:.4f}")

    # Ensure a mix of single words and phrases
    sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)
    phrases = [(term, score) for term, score in sorted_terms if '_' in term][:10]  # Up to 10 phrases
    single_words = [(term, score) for term, score in sorted_terms if '_' not in term][:10]  # Up to 10 single words
    final_terms = phrases + single_words
    final_terms = sorted(final_terms, key=lambda x: x[1], reverse=True)[:target_word_count]
    seed_words[category] = [term for term, _ in final_terms]
    category_word_scores[category] = term_scores
    logging.info(f"Category {category}: Top {len(seed_words[category])} seed words selected: {seed_words[category]}")

Deduplicate and Finalize Seed Words

In [8]:
logging.info("Started deduplicating seed words")
final_seed_words = {category: [] for category in seed_words}
word_to_category = {}
word_to_score = {}

all_terms = []
for category, words in seed_words.items():
    for word in words:
        score = category_word_scores[category].get(word, 0)
        all_terms.append((word, category, score))

all_terms.sort(key=lambda x: x[2], reverse=True)

for word, category, score in all_terms:
    if word not in word_to_category:
        final_seed_words[category].append(word)
        word_to_category[word] = category
        word_to_score[word] = score

for category in final_seed_words:
    current_words = final_seed_words[category]
    remaining_slots = target_word_count - len(current_words)
    if remaining_slots > 0:
        available_terms = [
            (term, score) for term, score in sorted(category_word_scores[category].items(), key=lambda x: x[1], reverse=True)
            if term not in word_to_category and score > 0.04
        ]
        final_seed_words[category].extend(term for term, _ in available_terms[:remaining_slots])
    final_seed_words[category] = final_seed_words[category][:target_word_count]

Validate and Save Seed Words

In [9]:
logging.info("Validating seed words")
print("\nSeed Words by Category:")
for category, words in final_seed_words.items():
    logging.info(f"Category: {category} -> Seed Words ({len(words)}): {words}")
    print(f"Category: {category} -> Seed Words ({len(words)}): {words}")

word_overlap = {}
for category, words in final_seed_words.items():
    for word in words:
        if word not in word_overlap:
            word_overlap[word] = []
        word_overlap[word].append(category)
overlap_words = {word: cats for word, cats in word_overlap.items() if len(cats) > 1}
if overlap_words:
    logging.warning("Overlapping words detected")
    print("\nWarning: Overlapping words detected:")
    for word, cats in overlap_words.items():
        logging.warning(f"Word '{word}' appears in {cats}")
        print(f"Word '{word}' appears in {cats}")
else:
    logging.info("No overlapping words detected")
    print("\nNo overlapping words detected.")

seed_words_df = pd.DataFrame([(cat, word) for cat, words in final_seed_words.items() for word in words], columns=['Category', 'SeedWord'])
seed_words_path = '../../datasets/seed_words.csv'
seed_words_df.to_csv(seed_words_path, index=False)
logging.info(f"Seed words saved to {seed_words_path}")
print(f"\nSeed words saved to {seed_words_path}")


Seed Words by Category:
Category: F -> Seed Words (20): ['collision_estimate', 'modify_relating', 'estimator_recycled', 'used_estimate', 'player', 'display', 'program', 'authentication_right', 'instructor_need', 'scheduler_cancel', 'retain_recycled', 'view_schedule', 'shopping_cart', 'request', 'student', 'view', 'list', 'site', 'formula_semi', 'posted_customer']
Category: FT -> Seed Words (20): ['report_appointment', 'high_number', 'warn_eca', 'fault', 'failure', 'reliability', 'database', 'loss', 'filesystems', 'tolerance', 'website', 'continue', 'crash', 'robust', 'tablet', 'malicious', 'around', 'overall', 'alive', 'transmission']
Category: PE -> Seed Words (20): ['take_min', 'depends_medium', 'display_confirmation', 'maximum_second', 'performance_degradation', 'application_concurrent', 'second_submits', 'website_minute', 'second_poll', 'second', 'response', 'minute', 'load', 'take', 'longer', 'application', 'connection', 'movie', 'process', 'maximum']
Category: PO -> Seed Words (

Load Seed Words

In [10]:
def load_seed_words_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        seed_words = df.groupby('Category')['SeedWord'].apply(list).to_dict()
        logging.info(f"Successfully loaded seed words from {file_path}")
        return seed_words
    except FileNotFoundError:
        logging.error(f"Seed words file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading seed words: {e}")
        raise

seed_words = load_seed_words_from_csv(seed_words_path)
seed_word_set = set(word for words in seed_words.values() for word in words)
stop_words = stop_words - seed_word_set

Define Preprocessing Function

In [11]:
def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens
              if word not in stop_words and len(word) > 2]
    return tokens

Load and Preprocess Review Data

In [12]:
input_path = '../../datasets/review_train.csv'
output_path = '../../datasets/selected_pseudo_labeled.csv'
threshold = 0.8
entropy_threshold = 1.0  # For filtering noisy samples

os.makedirs('models', exist_ok=True)

logging.info("Started loading review data")
df = pd.read_csv(input_path)
df = df[['processed_reviews']].copy()
df['processed_reviews'] = df['processed_reviews'].fillna("")
logging.info("Completed loading review data")

logging.info("Started preprocessing reviews")
tqdm.pandas()
tokenized_reviews = df['processed_reviews'].progress_apply(preprocess)
valid_indices = [i for i, tokens in enumerate(tokenized_reviews) if tokens]
tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]
if not tokenized_reviews:
    raise ValueError("No valid reviews after preprocessing.")
logging.info("Completed preprocessing reviews")
filtered_df = df.iloc[valid_indices].copy()
logging.info(f"Filtered DataFrame to {len(filtered_df)} valid reviews")

100%|██████████| 286518/286518 [00:09<00:00, 31154.59it/s]


Detect Bigrams

In [ ]:
logging.info("Started detecting bigrams")
sample_size = min(100000, len(tokenized_reviews))
sampled_reviews = random.sample(tokenized_reviews, sample_size) if sample_size < len(tokenized_reviews) else tokenized_reviews
bigram_model = Phrases(sampled_reviews, min_count=5, threshold=0.5, scoring='npmi')
bigram_phraser = Phraser(bigram_model)
logging.info("Started applying bigrams")
tokenized_reviews = [bigram_phraser[tokens] for tokens in tokenized_reviews]
logging.info("Completed applying bigrams")

Create Dictionary and Check Seed Words

In [14]:
logging.info("Started creating dictionary")
dictionary = corpora.Dictionary(tokenized_reviews)
dictionary.filter_extremes(no_below=1, no_above=0.8)
logging.info("Completed creating dictionary")

logging.info("Checking seed word presence in corpus")
all_seed_words = set(word for words in seed_words.values() for word in words)
# Corpus words should include bigram phrases as single tokens
corpus_words = set(word for review in tokenized_reviews for word in review)  # tokenized_reviews already has bigrams applied
missing_in_corpus = all_seed_words - corpus_words

if missing_in_corpus:
    print(f"WARNING: {len(missing_in_corpus)} seed words are NOT present in the corpus:")
    print("Missing seed words and suggested near-matches:")
    for word in sorted(list(missing_in_corpus)):
        # For phrases (containing '_'), compare against other phrases; for single words, compare against single words
        if '_' in word:
            # Only compare against phrases (tokens with '_') in corpus
            near_matches = [
                w for w in corpus_words
                if '_' in w and (
                    lemmatizer.lemmatize(word.replace('_', ' ')) in w.replace('_', ' ') or
                    w.replace('_', ' ') in lemmatizer.lemmatize(word.replace('_', ' '))
                )
            ][:5]
        else:
            # For single words, compare against single words (no '_')
            near_matches = [
                w for w in corpus_words
                if '_' not in w and (
                    lemmatizer.lemmatize(word) in w or w in lemmatizer.lemmatize(word)
                )
            ][:5]
        print(f"  - '{word}': Near-matches = {near_matches if near_matches else 'None'}")
    logging.warning(f"{len(missing_in_corpus)} seed words not found in corpus")
else:
    print("SUCCESS: All seed words are present in the corpus.")
print("--------------------------------------------------\n")

print("\n--- Checking Seed Word Survival in Dictionary ---")
vocabulary = set(dictionary.token2id.keys())
missing_words = all_seed_words - vocabulary
if missing_words:
    print(f"WARNING: {len(missing_words)} seed words are NOT in the dictionary after filtering and will be ignored:")
    print(sorted(list(missing_words))[:20])
    if len(missing_words) > 20:
        print(f"... and {len(missing_words) - 20} more.")
else:
    print("SUCCESS: All seed words survived the dictionary filtering process.")
print("--------------------------------------------------\n")

Missing seed words and suggested near-matches:
  - 'access_attempt': Near-matches = None
  - 'added_minute': Near-matches = None
  - 'application_concurrent': Near-matches = None
  - 'audit_report': Near-matches = None
  - 'authenticated_authorized': Near-matches = None
  - 'authentication_right': Near-matches = None
  - 'capable_processing': Near-matches = None
  - 'cater_simultaneous': Near-matches = None
  - 'collision_estimate': Near-matches = None
  - 'customer_installation': Near-matches = None
  - 'database': Near-matches = ['ba', 'as', 'aba', 'base', 'tab']
  - 'dba_two': Near-matches = None
  - 'depends_medium': Near-matches = None
  - 'different_type': Near-matches = None
  - 'display_confirmation': Near-matches = None
  - 'encrypted_database': Near-matches = None
  - 'end_may': Near-matches = None
  - 'estimator_recycled': Near-matches = None
  - 'familiar_website': Near-matches = None
  - 'feel_satisfied': Near-matches = None
  - 'filesystems': Near-matches = ['stem', 'es',

Validate Seed Words and Prepare ETA

In [15]:
logging.info("Started validating seed words")
seed_word_ids = {
    topic: [dictionary.token2id[word] for word in words if word in dictionary.token2id]
    for topic, words in seed_words.items()
}
for topic, ids in seed_word_ids.items():
    logging.info(f"Topic {topic}: {len(ids)}/{len(seed_words[topic])} seed words in dictionary")
    if not ids:
        logging.warning(f"No seed words for {topic} found in dictionary. Topic may be misaligned.")
logging.info("Completed validating seed words")

topic_name_to_id = {name: idx for idx, name in enumerate(seed_words.keys())}
print("\nTopic Name to Integer ID Mapping:")
print(topic_name_to_id)

num_topics = len(seed_words)
num_words = len(dictionary)
eta = np.ones((num_topics, num_words)) * 0.01
for topic_name, seed_word_ids_in_topic in seed_word_ids.items():
    topic_id = topic_name_to_id[topic_name]
    for word_id in seed_word_ids_in_topic:
        eta[topic_id][word_id] = 500.0


Topic Name to Integer ID Mapping:
{'F': 0, 'FT': 1, 'PE': 2, 'PO': 3, 'SC': 4, 'SE': 5, 'US': 6}


Train LDA Model

In [16]:
logging.info("Started creating BoW corpus")
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]
logging.info("Completed creating BoW corpus")

logging.info("Started training LDA model")
print("\nTraining LDA model...")
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    passes=20,
    alpha=0.01/num_topics,
    iterations=400,
    eta=eta,
    random_state=42,
    minimum_probability=0.05,
    per_word_topics=True,
    decay=0.7,
    offset=50.0
)
logging.info("Completed training LDA model")
print("Training complete.")


Training LDA model...
Training complete.


Save Models

In [17]:
logging.info("Saving LDA model, dictionary, and bigram model")
lda_model.save('models/lda_model')
dictionary.save('models/dictionary')
bigram_phraser.save('models/bigram_phraser')
logging.info("Saved LDA model, dictionary, and bigram models")

Analyze Topics and Coherence

In [18]:
logging.info("Computing per-topic coherence scores")
print("\n--- Per-Topic Coherence Scores ---")
coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_reviews,
    dictionary=dictionary,
    coherence='c_v',
    topn=10,
    window_size=50
)
per_topic_coherence = coherence_model.get_coherence_per_topic()
id_to_topic_name = {v: k for k, v in topic_name_to_id.items()}
for i, score in enumerate(per_topic_coherence):
    topic_name = id_to_topic_name[i]
    logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f}")
    print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
print("------------------------------------\n")

logging.info("Discovered Topics (Top 30 words):")
print("\n--- Deeper Dive into Discovered Topics (Top 30 words) ---")
for i in range(num_topics):
    topic_name = id_to_topic_name[i]
    topic_words_probs = lda_model.show_topic(i, topn=30)
    topic_words = [word for word, prob in topic_words_probs]
    log_message = f"Topic #{i} ({topic_name}): {', '.join(topic_words)}"
    logging.info(log_message)
    print(f"\nTopic #{i}: {topic_name}")
    print(topic_words)
print("----------------------------------------------------------\n")


--- Per-Topic Coherence Scores ---
Topic #0: F - C_v Score = 0.7186
Topic #1: FT - C_v Score = 0.3314
Topic #2: PE - C_v Score = 0.5338
Topic #3: PO - C_v Score = 0.5164
Topic #4: SC - C_v Score = 0.5810
Topic #5: SE - C_v Score = 0.4496
Topic #6: US - C_v Score = 0.5250
------------------------------------


--- Deeper Dive into Discovered Topics (Top 30 words) ---

Topic #0: F
['little', 'need', 'assignment', 'however', 'video', 'better', 'difficult', 'dont', 'hard', 'though', 'code', 'know', 'didnt', 'still', 'overall', 'end', 'last', 'enough', 'feel', 'complete', 'quiz', 'found', 'final', 'quite', 'felt', 'find', 'reading', 'give', 'problem', 'sometimes']

Topic #1: FT
['learned', 'interesting', 'informative', 'fun', 'excel', 'love', 'excellent', 'continue', 'learnt', 'follow', 'feel', 'forward', 'looking_forward', 'lot', 'made', 'look', 'overall', 'super', 'taking', 'interactive', 'specialization', 'found', 'experience', 'happy', 'professor', 'definitely', 'keep', 'organized', 'e

Verify Seed Word Probabilities (Detailed)

In [19]:
print("\n--- Verifying Seed Word Probabilities and Ranks in Final Model ---")
logging.info("Verifying Seed Word Probabilities and Ranks in Final Model")
for topic_name, words in seed_words.items():
    topic_id = topic_name_to_id[topic_name]
    print(f"\nAssigned Topic: {topic_name} (ID: {topic_id})")
    logging.info(f"Assigned Topic: {topic_name} (ID: {topic_id})")
    
    for word in words:
        if word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            term_topics = lda_model.get_term_topics(word_id, minimum_probability=0.0)
            print(f"  - Seed Word '{word}'")
            logging.info(f"  - Seed Word '{word}'")
            for t_id, prob in term_topics:
                t_name = id_to_topic_name[t_id]
                topic_words_probs = lda_model.show_topic(t_id, topn=len(dictionary))
                word_to_rank = {w: idx + 1 for idx, (w, _) in enumerate(topic_words_probs)}
                rank = word_to_rank.get(word, "N/A")
                log_message = f"    * Topic {t_name} (ID: {t_id}): Probability = {prob:.4f}, Rank = {rank}"
                logging.info(log_message)
                print(log_message)
        else:
            logging.warning(f"Seed Word '{word}' for topic '{topic_name}' was not in the final dictionary.")
            print(f"  - '{word}' (Not in dictionary)")
print("----------------------------------------------------------\n")


--- Verifying Seed Word Probabilities and Ranks in Final Model ---

Assigned Topic: F (ID: 0)
  - 'collision_estimate' (Not in dictionary)
  - 'modify_relating' (Not in dictionary)
  - 'estimator_recycled' (Not in dictionary)
  - 'used_estimate' (Not in dictionary)
  - Seed Word 'player'
    * Topic F (ID: 0): Probability = 0.0008, Rank = 324
  - Seed Word 'display'
    * Topic F (ID: 0): Probability = 0.0009, Rank = 308
  - Seed Word 'program'
    * Topic F (ID: 0): Probability = 0.0031, Rank = 56
    * Topic FT (ID: 1): Probability = 0.0021, Rank = 115
    * Topic SC (ID: 4): Probability = 0.0006, Rank = 353
    * Topic SE (ID: 5): Probability = 0.0019, Rank = 114
  - 'authentication_right' (Not in dictionary)
  - 'instructor_need' (Not in dictionary)
  - 'scheduler_cancel' (Not in dictionary)
  - 'retain_recycled' (Not in dictionary)
  - 'view_schedule' (Not in dictionary)
  - 'shopping_cart' (Not in dictionary)
  - Seed Word 'request'
    * Topic F (ID: 0): Probability = 0.0012, R

Compute Coherence Scores

In [20]:
logging.info("Started computing c_v coherence scores")
window_size = 50

print("\n--- Computing C_v Coherence ---")
logging.info("Computing c_v coherence")

coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_reviews,
    dictionary=dictionary,
    coherence='c_v',
    topn=10,
    window_size=window_size
)

per_topic_coherence = coherence_model.get_coherence_per_topic()
print("\nPer-Topic C_v Scores:")
for i, score in enumerate(per_topic_coherence):
    topic_name = id_to_topic_name[i]
    logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f}")
    print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")

coherence_score = coherence_model.get_coherence()
logging.info(f"Overall C_v Coherence Score: {coherence_score:.4f}")
print(f"\nOverall C_v Coherence Score: {coherence_score:.4f}")
print("------------------------------------\n")


--- Computing C_v Coherence ---

Per-Topic C_v Scores:
Topic #0: F - C_v Score = 0.7186
Topic #1: FT - C_v Score = 0.3314
Topic #2: PE - C_v Score = 0.5338
Topic #3: PO - C_v Score = 0.5164
Topic #4: SC - C_v Score = 0.5810
Topic #5: SE - C_v Score = 0.4496
Topic #6: US - C_v Score = 0.5250

Overall C_v Coherence Score: 0.5222
------------------------------------



In [21]:
def compute_coherence_range(lda_model, tokenized_reviews, dictionary, window_size=50, topn_range=(10, 100, 10)):
    """
    Compute C_v coherence scores for a range of topn values.
    
    Parameters:
    - lda_model: Trained LDA model
    - tokenized_reviews: List of tokenized texts
    - dictionary: Gensim dictionary
    - window_size: Window size for coherence calculation
    - topn_range: Tuple of (start, end, step) for topn values
    """
    logging.info("Started computing C_v coherence scores for multiple topn values")
    print("\n--- Computing C_v Coherence for Multiple topn Values ---")
    
    start, end, step = topn_range
    coherence_results = []
    
    for topn in range(start, end + 1, step):
        logging.info(f"Computing C_v coherence for topn={topn}")
        print(f"\nComputing C_v Coherence for topn={topn}")
        
        coherence_model = CoherenceModel(
            model=lda_model,
            texts=tokenized_reviews,
            dictionary=dictionary,
            coherence='c_v',
            topn=topn,
            window_size=window_size
        )
        
        per_topic_coherence = coherence_model.get_coherence_per_topic()
        print(f"\nPer-Topic C_v Scores (topn={topn}):")
        for i, score in enumerate(per_topic_coherence):
            topic_name = id_to_topic_name[i]
            logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f} (topn={topn})")
            print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
        
        coherence_score = coherence_model.get_coherence()
        logging.info(f"Overall C_v Coherence Score (topn={topn}): {coherence_score:.4f}")
        print(f"\nOverall C_v Coherence Score (topn={topn}): {coherence_score:.4f}")
        print("-" * 35)
        
        coherence_results.append({
            'topn': topn,
            'per_topic_coherence': per_topic_coherence,
            'overall_coherence': coherence_score
        })
    
    logging.info("Completed computing C_v coherence scores for multiple topn values")
    print("\n--- Completed Computing C_v Coherence for Multiple topn Values ---")
    return coherence_results

# Call the new function
coherence_results = compute_coherence_range(
    lda_model=lda_model,
    tokenized_reviews=tokenized_reviews,
    dictionary=dictionary,
    window_size=50,
    topn_range=(10, 100, 10)
)


--- Computing C_v Coherence for Multiple topn Values ---

Computing C_v Coherence for topn=10

Per-Topic C_v Scores (topn=10):
Topic #0: F - C_v Score = 0.7186
Topic #1: FT - C_v Score = 0.3314
Topic #2: PE - C_v Score = 0.5338
Topic #3: PO - C_v Score = 0.5164
Topic #4: SC - C_v Score = 0.5810
Topic #5: SE - C_v Score = 0.4496
Topic #6: US - C_v Score = 0.5250

Overall C_v Coherence Score (topn=10): 0.5222
-----------------------------------

Computing C_v Coherence for topn=20

Per-Topic C_v Scores (topn=20):
Topic #0: F - C_v Score = 0.7516
Topic #1: FT - C_v Score = 0.3696
Topic #2: PE - C_v Score = 0.5425
Topic #3: PO - C_v Score = 0.4093
Topic #4: SC - C_v Score = 0.5198
Topic #5: SE - C_v Score = 0.4500
Topic #6: US - C_v Score = 0.4779

Overall C_v Coherence Score (topn=20): 0.5030
-----------------------------------

Computing C_v Coherence for topn=30

Per-Topic C_v Scores (topn=30):
Topic #0: F - C_v Score = 0.7878
Topic #1: FT - C_v Score = 0.4135
Topic #2: PE - C_v Score 

Assign Topics to Reviews and Save with Topic Probabilities

In [22]:
logging.info("Started getting topic distributions")
doc_topics = [lda_model.get_document_topics(doc, minimum_probability=0.0) for doc in corpus]
topic_matrix = np.zeros((len(corpus), num_topics))
for i, topics in enumerate(doc_topics):
    for topic_id, prob in topics:
        topic_matrix[i, topic_id] = prob
logging.info("Completed getting topic distributions")

logging.info("Started saving all reviews with topics")
all_topics_df = filtered_df.copy()
all_topics_df['Topic'] = np.argmax(topic_matrix, axis=1)
all_topics_df['topic_name'] = all_topics_df['Topic'].map({v: k for k, v in topic_name_to_id.items()})
all_topics_path = '../../datasets/reviews_with_topic.csv'
all_topics_df[['processed_reviews', 'Topic', 'topic_name']].to_csv(all_topics_path, index=False)
logging.info(f"All reviews with topic assignments saved to {all_topics_path}")
print(f"\nAll reviews with topic assignments saved to {all_topics_path}")

logging.info("Sample of saved data:")
logging.info(all_topics_df[['processed_reviews', 'Topic', 'topic_name']].head(2).to_string())
print("Sample of saved data:")
print(all_topics_df[['processed_reviews', 'Topic', 'topic_name']].head(2))


All reviews with topic assignments saved to ../../datasets/reviews_with_topic.csv
Sample of saved data:
                                   processed_reviews  Topic topic_name
0  thank much lot good find course like especiall...      2         PE
1  reading many negative true certain becomes wai...      0          F


Select and Save Pseudo-Labeled Reviews with Topic Probabilities and Entropy Filtering

In [23]:
logging.info(f"Started selecting reviews with confidence > {threshold} and entropy <= {entropy_threshold}")
selected_reviews = {topic_name: [] for topic_name in seed_words.keys()}
non_selected_reviews = {topic_name: [] for topic_name in seed_words.keys()}
id_to_topic = {v: k for k, v in topic_name_to_id.items()}

# Compute topic entropy for all reviews
entropies = [entropy(probs) for probs in topic_matrix]
logging.info(f"Average topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")
print(f"\nAverage topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")

for i, (review_text, topic_probs) in enumerate(zip(filtered_df['processed_reviews'], topic_matrix)):
    dominant_topic = np.argmax(topic_probs)
    dominant_prob = topic_probs[dominant_topic]
    topic_name = id_to_topic[dominant_topic]
    review_entropy = entropy(topic_probs)
    review_data = {
        'original_index': valid_indices[i],
        'topic': topic_name,
        'confidence': dominant_prob,
        'text': review_text,
        'topic_probs': ','.join(map(str, topic_probs))
    }
    if dominant_prob > threshold and review_entropy <= entropy_threshold:
        selected_reviews[topic_name].append(review_data)
    else:
        non_selected_reviews[topic_name].append(review_data)
logging.info("Completed selecting reviews")

# Log and print summary
logging.info("High-Confidence Reviews Summary:")
print("\nHigh-Confidence Reviews Summary:")
for topic_name, reviews in selected_reviews.items():
    logging.info(f"Topic {topic_name}: {len(reviews):,} high-confidence reviews selected")
    print(f"Topic {topic_name}: {len(reviews):,} high-confidence reviews selected")

logging.info("Non-High-Confidence Reviews Summary:")
print("\nNon-High-Confidence Reviews Summary:")
for topic_name, reviews in non_selected_reviews.items():
    logging.info(f"Topic {topic_name}: {len(reviews):,} non-high-confidence reviews selected")
    print(f"Topic {topic_name}: {len(reviews):,} non-high-confidence reviews selected")

# Save high-confidence reviews
logging.info("Started saving high-confidence reviews")
selected_data = []
for topic_name, reviews in selected_reviews.items():
    for rev in reviews:
        selected_data.append({
            'original_index': rev['original_index'],
            'topic': rev['topic'],
            'confidence': rev['confidence'],
            'text': rev['text'],
            'topic_probs': rev['topic_probs']
        })
selected_df = pd.DataFrame(selected_data)
selected_df.to_csv(output_path, index=False)
logging.info(f"High-confidence pseudo-labeled reviews saved to {output_path}")
print(f"\nHigh-confidence pseudo-labeled reviews saved to {output_path}")

# Save non-high-confidence reviews
logging.info("Started saving non-high-confidence reviews")
non_selected_data = []
for topic_name, reviews in non_selected_reviews.items():
    for rev in reviews:
        non_selected_data.append({
            'original_index': rev['original_index'],
            'text': rev['text']
        })
non_selected_df = pd.DataFrame(non_selected_data)
non_selected_path = '../../datasets/non_high_confidence_reviews.csv'
non_selected_df.to_csv(non_selected_path, index=False)
logging.info(f"Non-high-confidence reviews saved to {non_selected_path}")
print(f"\nNon-high-confidence reviews saved to {non_selected_path}")


Average topic entropy: 0.6336, Std: 0.4491

High-Confidence Reviews Summary:
Topic F: 20,529 high-confidence reviews selected
Topic FT: 14,117 high-confidence reviews selected
Topic PE: 8,871 high-confidence reviews selected
Topic PO: 6,526 high-confidence reviews selected
Topic SC: 11,030 high-confidence reviews selected
Topic SE: 18,171 high-confidence reviews selected
Topic US: 12,576 high-confidence reviews selected

Non-High-Confidence Reviews Summary:
Topic F: 48,227 non-high-confidence reviews selected
Topic FT: 21,390 non-high-confidence reviews selected
Topic PE: 26,682 non-high-confidence reviews selected
Topic PO: 14,509 non-high-confidence reviews selected
Topic SC: 27,402 non-high-confidence reviews selected
Topic SE: 26,876 non-high-confidence reviews selected
Topic US: 24,590 non-high-confidence reviews selected

High-confidence pseudo-labeled reviews saved to ../../datasets/selected_pseudo_labeled.csv

Non-high-confidence reviews saved to ../../datasets/non_high_confid